In [17]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

In [18]:
input_csv = Path("/Users/andrewweiland/Chopra_Lab_Research/multi_objective_chemical_selection/curated_200_approved_drug_molecules.csv"
)
compounds = pd.read_csv(input_csv)
if "smiles" not in compounds.columns:
    raise ValueError("The input CSV must contain a 'smiles' column.")
print(f"Loaded {len(compounds)} compounds")
compounds.head()

Loaded 200 compounds


,candidate_id,chembl_id,preferred_name,smiles,standard_inchi_key,molecular_formula,molecular_weight,alogp,polar_surface_area,hbond_acceptors,...,first_approval_year,max_phase,oral,parenteral,topical,prodrug,black_box_warning,source_database,source_record_url,source_retrieved_date
0,MOO001,CHEMBL2205807,ABAMETAPIR,Cc1ccc(-c2ccc(C)cn2)nc1,PTRATZCAGVBFIQ-UHFFFAOYSA-N,C12H12N2,184.24,2.76,25.78,2,...,2020.0,4,False,False,True,False,False,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29
1,MOO002,CHEMBL271227,ABIRATERONE ACETATE,CC(=O)O[C@H]1CC[C@@]2(C)C(=CC[C@@H]3[C@@H]2CC[...,UVIQSJCZCSLXRZ-UBUQANBQSA-N,C26H33NO2,391.56,5.97,39.19,3,...,2011.0,4,True,False,False,True,False,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29
2,MOO003,CHEMBL1201327,ACETRIZOIC ACID,CC(=O)Nc1c(I)cc(I)c(C(=O)O)c1I,GNOGSFBXBWBTIG-UHFFFAOYSA-N,C9H6I3NO3,556.86,3.16,66.40,2,...,1982.0,4,False,False,True,False,False,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29
3,MOO004,CHEMBL3940890,ACORAMIDIS,Cc1n[nH]c(C)c1CCCOc1cc(C(=O)O)ccc1F,WBFUHHBPNXWNCC-UHFFFAOYSA-N,C15H17FN2O3,292.31,2.88,75.21,3,...,2024.0,4,True,False,False,False,False,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29
4,MOO005,CHEMBL634,ALFENTANIL,CCC(=O)N(c1ccccc1)C1(COC)CCN(CCn2nnn(CC)c2=O)CC1,IDBPHNDTYPBSNI-UHFFFAOYSA-N,C21H32N6O3,416.53,1.38,85.49,8,...,1986.0,4,False,True,False,False,True,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29


In [30]:
# Change these two values for the microservice you want to test.
microservice_port = 8000
property_name = "solubility"

microservice_url = f"http://localhost:{microservice_port}"
pause_seconds = 0.2
connect_timeout_seconds = 10
read_timeout_seconds = 60
max_consecutive_failures = 3

print(f"Testing {property_name!r} at {microservice_url}")

Testing 'lipophilicity' at http://localhost:8000


In [31]:
def check_health(timeout=10):
    response = requests.get(f"{microservice_url}/health", timeout=timeout)
    response.raise_for_status()
    return response.json()

check_health()

{'status': 'healthy', 'message': 'Admetica microservice is running'}

In [32]:
test_smiles = str(compounds.iloc[0]["smiles"])
test_response = requests.post(
    f"{microservice_url}/smi",
    json={"smiles": test_smiles, "property": [property_name]},
    timeout=(connect_timeout_seconds, read_timeout_seconds),
)
print(test_response.status_code)
test_response.json()

200


{'smiles': 'Cc1ccc(-c2ccc(C)cn2)nc1',
 'status': 'success',
 'results': {'lipophilicity': {'property': 'lipophilicity',
   'status': 'success',
   'prediction': 2.1108028888702393,
   'error': None}},
 'error': None}

In [22]:
def extract_prediction(body, expected_property):
    top_status = str(
        body.get("status", "")
    ).lower()

    if top_status != "success":
        raise RuntimeError(
            f"Microservice returned an error: {body}"
        )

    results = body.get("results", {})
    result = results.get(expected_property)

    if result is None:
        available = list(results)

        raise KeyError(
            f"Property {expected_property!r} was not returned; "
            f"available: {available}"
        )

    property_status = str(
        result.get("status", "")
    ).lower()

    if property_status != "success":
        error_message = result.get(
            "error",
            "Unknown property prediction error",
        )

        raise RuntimeError(
            f"Prediction of {expected_property!r} failed: "
            f"{error_message}"
        )

    value = result.get("results")

    if value is None:
        value = result.get("prediction")

    if value is None:
        raise ValueError(
            f"Property {expected_property!r} reported success "
            "but returned no numeric value."
        )

    return float(value)

In [23]:
session = requests.Session()

def query_microservice(smiles):
    started = time.perf_counter()
    response = session.post(
        f"{microservice_url}/smi",
        json={"smiles": smiles, "property": [property_name]},
        timeout=(connect_timeout_seconds, read_timeout_seconds),
    )
    elapsed = time.perf_counter() - started
    response.raise_for_status()
    body = response.json()
    prediction = extract_prediction(body, property_name)
    return prediction, elapsed, body

In [24]:
test_prediction, test_elapsed, test_body = query_microservice(test_smiles)
print(f"Prediction: {test_prediction}")
print(f"Response time: {test_elapsed:.3f} seconds")
test_body

KeyError: "Property 'LogP' was not returned; available: []"

In [28]:
response = requests.post(
    f"{microservice_url}/smi",
    json={
        "smiles": test_smiles,
        "property": [property_name],
    },
    timeout=(10, 30),
)

print("HTTP status:", response.status_code)

test_body = response.json()

print(json.dumps(test_body, indent=2))

HTTP status: 200
{
  "smiles": "Cc1ccc(-c2ccc(C)cn2)nc1",
  "status": "success",
  "results": {
    "Lipophilicity_AstraZeneca": {
      "property": "Lipophilicity_AstraZeneca",
      "status": "success",
      "results": 2.2844,
      "error": null
    }
  },
  "error": null
}


In [33]:
request_rows = []
consecutive_failures = 0

for position, (_, compound) in enumerate(compounds.iterrows(), start=1):
    smiles = str(compound["smiles"])
    print(f"{position}/{len(compounds)}: {smiles}", flush=True)
    started = time.perf_counter()
    try:
        prediction, elapsed, body = query_microservice(smiles)
        request_rows.append({
            "request_number": position,
            "smiles": smiles,
            "property": property_name,
            "prediction": prediction,
            "response_seconds": elapsed,
            "status": "success",
            "error": None,
            "raw_response": json.dumps(body),
        })
        consecutive_failures = 0
        print(f"  success in {elapsed:.3f}s", flush=True)
    except Exception as exc:
        elapsed = time.perf_counter() - started
        consecutive_failures += 1
        request_rows.append({
            "request_number": position,
            "smiles": smiles,
            "property": property_name,
            "prediction": np.nan,
            "response_seconds": elapsed,
            "status": "error",
            "error": str(exc),
            "raw_response": None,
        })
        print(f"  ERROR after {elapsed:.3f}s: {exc}", flush=True)
        if consecutive_failures >= max_consecutive_failures:
            print("Stopping after repeated failures.", flush=True)
            break
    time.sleep(pause_seconds)

1/200: Cc1ccc(-c2ccc(C)cn2)nc1
  success in 0.023s
2/200: CC(=O)O[C@H]1CC[C@@]2(C)C(=CC[C@@H]3[C@@H]2CC[C@]2(C)C(c4cccnc4)=CC[C@@H]32)C1
  success in 0.036s
3/200: CC(=O)Nc1c(I)cc(I)c(C(=O)O)c1I
  success in 0.028s
4/200: Cc1n[nH]c(C)c1CCCOc1cc(C(=O)O)ccc1F
  success in 0.023s
5/200: CCC(=O)N(c1ccccc1)C1(COC)CCN(CCn2nnn(CC)c2=O)CC1
  success in 0.031s
6/200: COc1cc2nc(N(C)CCCNC(=O)C3CCCO3)nc(N)c2cc1OC
  success in 0.032s
7/200: CN(C)c1nc(N(C)C)nc(N(C)C)n1
  success in 0.030s
8/200: CC[N+](CC)(CCNC(=O)C(=O)NCC[N+](CC)(CC)Cc1ccccc1Cl)Cc1ccccc1Cl
  success in 0.029s
9/200: CC(C)(C)C(=O)OCOC(=O)[C@@H]1N2C(=O)[C@@H](/N=C/N3CCCCCC3)[C@H]2SC1(C)C
  success in 0.033s
10/200: N=C(N)NC(=O)c1nc(Cl)c(N)nc1N
  success in 0.029s


KeyboardInterrupt: 